# Fase 0 — Setup do ambiente

Preparação do ambiente de execução do estudo comparativo **U-Net (CNN) vs SegFormer (ViT)** para segmentação semântica de cafezais.

Este estágio detecta a plataforma de execução, instala as dependências necessárias, carrega a configuração única do projeto, fixa as sementes, autentica o Google Earth Engine e registra o ambiente do run. As saídas são: ambiente pronto, configuração carregada e credenciais do GEE validadas.

## Obtenção do repositório

Garante a presença do pacote `src/` numa área gravável da plataforma: no Colab clona o repositório público para `/content` e atualiza em execuções seguintes via `git pull`; no Kaggle copia o dataset somente-leitura de `/kaggle/input` para `/kaggle/working` (ou clona do GitHub se não houver dataset). No ambiente local a etapa é ignorada, pois o repositório já está no diretório corrente.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
from pathlib import Path


# URL pública do repositório, usada como fallback quando não há dataset montado.
REPO_URL = "https://github.com/jotap1101/tcc.git"
REPO_NAME = "tcc"


def _copy_repo(source: Path, dest: Path) -> None:
    """Copia o repositório para a área gravável, ignorando metadados de versionamento."""
    dest.mkdir(parents=True, exist_ok=True)
    for item in source.iterdir():
        if item.name in {".git", "__pycache__", ".ruff_cache", ".mypy_cache", ".pytest_cache"}:
            continue
        target = dest / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)


# Detecta Colab com segurança, mesmo quando o pacote google não existe.
is_colab = "COLAB_GPU" in os.environ
if not is_colab:
    try:
        is_colab = importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        is_colab = False

# Resolve a área gravável e a fonte do repositório conforme a plataforma.
if is_colab:
    working_dir = Path("/content") / REPO_NAME
    repo_source = None
elif Path("/kaggle").is_dir():
    working_dir = Path("/kaggle/working") / REPO_NAME
    repo_source = next(
        (p for p in Path("/kaggle/input").glob("*") if (p / "src" / "config.yaml").is_file()),
        None,
    )
else:
    working_dir = None
    repo_source = None

# Sincroniza o repositório: atualiza o clone via git pull, copia o dataset ou clona do GitHub.
def _sync_repo(working_dir: Path, repo_source: Path | None) -> None:
    if (working_dir / ".git").is_dir():
        subprocess.run(["git", "-C", str(working_dir), "pull", "--ff-only"], check=True)
        print(f"Repositório atualizado em {working_dir}.")
    elif repo_source is not None:
        _copy_repo(repo_source, working_dir)
        print(f"Repositório copiado de {repo_source} para {working_dir}.")
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(working_dir)], check=True)
        print(f"Repositório clonado em {working_dir}.")


if working_dir is not None:
    _sync_repo(working_dir, repo_source)
    os.environ["TCC_ROOT"] = str(working_dir)
else:
    print("Ambiente local: repositório já disponível no diretório corrente.")

## Detecção da raiz do repositório

Localiza a raiz do repositório pelo marcador `src/config.yaml` nos diretórios corrente, ancestrais e raízes de montagem das plataformas de nuvem (Kaggle/Colab), além do override via variável de ambiente `TCC_ROOT`. A raiz é inserida no caminho de importação, garantindo o acesso ao pacote `src/`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


# Raízes de busca: env TCC_ROOT, diretório corrente com ancestrais e montagens
# de repositórios nas plataformas de nuvem (Kaggle /kaggle/input, Colab /content).
def _candidate_bases() -> list[Path]:
    bases: list[Path] = []
    tcc_root = os.environ.get("TCC_ROOT")
    if tcc_root:
        bases.append(Path(tcc_root))
    bases.extend([Path.cwd(), *Path.cwd().parents])
    for mount in (Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")):
        if mount.is_dir():
            bases.append(mount)
    return bases


# Verifica a base e seus subdiretórios imediatos em busca do marcador da raiz.
def _find_project_root() -> Path:
    for base in _candidate_bases():
        for candidate in [base, *base.glob("*")]:
            if candidate.is_dir() and (candidate / "src" / "config.yaml").is_file():
                return candidate
    raise RuntimeError("Raiz do repositório não localizada (src/config.yaml ausente).")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Raiz do projeto: {PROJECT_ROOT}")

## Detecção da plataforma

Identifica o ambiente de execução (Kaggle, Colab ou local) para adaptar a instalação de dependências e a leitura de segredos.

In [ ]:
import importlib.util
import os


def _is_colab_runtime() -> bool:
    """Detecta Colab com segurança, mesmo quando o pacote google não existe."""
    if "COLAB_GPU" in os.environ:
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


# Colab é detectado primeiro, pois /kaggle também existe nos runtimes do Colab.
def detect_platform() -> str:
    if _is_colab_runtime():
        return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir():
        return "kaggle"
    return "local"


PLATFORM = detect_platform()
print(f"Plataforma detectada: {PLATFORM}")

## Instalação condicional das dependências

Em Kaggle/Colab tenta instalar o pacote com os extras geoespaciais e de aprendizado de máquina. Se a instalação falhar, o notebook continua, pois `src/` é importável via `sys.path` e as dependências pesadas já vêm instaladas nas plataformas. No ambiente local a instalação é ignorada, pois é gerenciada por `uv` e pelo CI.

In [ ]:
import subprocess

# Instala o projeto editavelmente com os extras necessários apenas em nuvem.
# Em falha, apenas avisa: src/ continua importável via sys.path já adicionado.
if PLATFORM in {"kaggle", "colab"}:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[geo,ml]"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("Aviso: pip install -e .[geo,ml] falhou (últimas linhas abaixo).")
        print("O pacote src/ segue importável via sys.path; se o erro for de versão")
        print("do Python, reexecute a célula 'Obtenção do repositório' para sincronizar.")
        print(result.stderr[-2000:])
    else:
        print("Dependências instaladas.")
else:
    print("Ambiente local: instalação ignorada (gerenciada por uv/CI).")

## Carregamento da configuração única

Lê a configuração de `src/config.yaml` por meio de `src/config.py`, fonte única de verdade de caminhos, bandas, parâmetros e sementes.

In [ ]:
from src.config import CONFIG

# Exibe um resumo dos parâmetros centrais do estudo.
print(f"Projeto: {CONFIG.get('project.name')} v{CONFIG.get('project.version')}")
print(f"Semente: {CONFIG.seed} | Patch: {CONFIG.patch_size} | Folds: {CONFIG.fold_count}")
print(f"Bandas: {CONFIG.bands}")
print(f"Hash da configuração: {CONFIG.config_hash}")

## Criação da árvore de diretórios

Garante a existência dos diretórios de dados, modelos e artefatos resolvidos a partir da configuração.

In [ ]:
# Cria os diretórios de trabalho, se ainda não existirem.
CONFIG.paths.ensure()
for name in ("data", "raw", "interim", "processed", "external", "models", "artifacts"):
    print(f"{name}: {getattr(CONFIG.paths, name)}")

## Fixação das sementes

Fixa as sementes de `python`, `numpy`, `torch` e `cuda` para garantir a reprodutibilidade dos experimentos.

In [ ]:
from src.config import seed_everything

# Aplica a semente global definida na configuração.
resolved_seed = seed_everything()
print(f"Sementes fixadas em {resolved_seed}.")

## Carregamento de segredos

Em Kaggle/Colab os segredos são cadastrados no cofre da plataforma (Colab: painel Segredos na barra lateral; Kaggle: Add-ons → Secrets) e injetados nas variáveis de ambiente esperadas pelo pacote. Nenhum valor é impresso. No ambiente local, os segredos devem vir de variáveis de ambiente ou do arquivo `.env`.

A autenticação do Earth Engine aceita dois modos: conta de serviço (`GEE_SERVICE_ACCOUNT_EMAIL` + `GEE_SERVICE_ACCOUNT_KEY_JSON`) ou OAuth de usuário (`GEE_OAUTH_CREDENTIALS_JSON`). `GEE_PROJECT` é comum e obrigatório. O modo ativo é o que tiver os segredos completos cadastrados no cofre.

In [ ]:
# Nomes das variáveis de ambiente consumidas pelas fases seguintes.
GEE_COMMON_SECRETS = ("GEE_PROJECT",)
SERVICE_ACCOUNT_SECRETS = ("GEE_SERVICE_ACCOUNT_EMAIL", "GEE_SERVICE_ACCOUNT_KEY_JSON")
OAUTH_SECRETS = ("GEE_OAUTH_CREDENTIALS_JSON",)
OPTIONAL_SECRETS = ("HF_TOKEN", "HF_USERNAME")


def _load_secret(name: str) -> str:
    # Lê do cofre da plataforma (Colab ou Kaggle) e retorna o valor do segredo.
    if PLATFORM == "kaggle":
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    from google.colab import userdata

    return userdata.get(name)


if PLATFORM in {"kaggle", "colab"}:
    if PLATFORM == "colab":
        print("Colab: cadastre os segredos no painel Segredos da barra lateral esquerda.")
    else:
        print("Kaggle: cadastre os segredos em Add-ons -> Secrets.")

    loaded: set[str] = set()
    for name in GEE_COMMON_SECRETS + SERVICE_ACCOUNT_SECRETS + OAUTH_SECRETS + OPTIONAL_SECRETS:
        try:
            os.environ[name] = _load_secret(name)
            loaded.add(name)
        except Exception:
            pass

    for name in OPTIONAL_SECRETS:
        if name not in loaded:
            print(f"Segredo opcional ausente: {name}")

    if "GEE_PROJECT" not in loaded:
        print("Segredo obrigatório ausente: GEE_PROJECT")
        print("Sem ele, a autenticação no Earth Engine falhará na célula seguinte.")
    elif set(SERVICE_ACCOUNT_SECRETS).issubset(loaded):
        print("Autenticação GEE: conta de serviço (service account).")
    elif set(OAUTH_SECRETS).issubset(loaded):
        print("Autenticação GEE: OAuth de usuário.")
    else:
        print("Modo de autenticação GEE incompleto: cadastre a conta de serviço "
              "(GEE_SERVICE_ACCOUNT_EMAIL + GEE_SERVICE_ACCOUNT_KEY_JSON) ou o "
              "OAuth (GEE_OAUTH_CREDENTIALS_JSON).")
else:
    print("Ambiente local: segredos esperados via variáveis de ambiente/.env.")

### Geração opcional das credenciais OAuth

No Colab, as credenciais OAuth podem ser geradas pelo fluxo oficial do Earth Engine. Execute a célula abaixo apenas se for usar OAuth e o segredo `GEE_OAUTH_CREDENTIALS_JSON` ainda não estiver cadastrado: ela abre a janela de autorização do Google, salva o arquivo de credenciais e indica o caminho. Copie o conteúdo do arquivo e cadastre-o no painel Segredos como `GEE_OAUTH_CREDENTIALS_JSON`, em uma única linha.

In [ ]:
# Gera as credenciais OAuth do Earth Engine via fluxo do Google (apenas no Colab).
if PLATFORM == "colab" and "GEE_OAUTH_CREDENTIALS_JSON" not in os.environ:
    import ee

    ee.Authenticate()  # abre o fluxo de autorização no navegador do Colab
    creds_path = Path.home() / ".config" / "earthengine" / "credentials"
    print("Credenciais OAuth geradas em:", creds_path)
    print("Copie o conteúdo do arquivo para o segredo GEE_OAUTH_CREDENTIALS_JSON.")
else:
    print("Nada a fazer: OAuth já configurado ou plataforma sem fluxo interativo.")

## Autenticação no Google Earth Engine

Inicializa o Earth Engine com as credenciais lidas exclusivamente do ambiente: conta de serviço ou credenciais OAuth de usuário. A ausência de credenciais é reportada sem interromper a execução do notebook.

In [ ]:
from src.data.gee_client import GEECredentialsError, init_ee

# Inicializa o cliente do Earth Engine a partir das variáveis de ambiente.
try:
    ee = init_ee()
    GEE_READY = True
    print("Earth Engine autenticado com sucesso.")
except GEECredentialsError as exc:
    GEE_READY = False
    print(f"Credenciais do GEE ausentes: {exc}")

## Registro do ambiente do run

Persiste versões de dependências, plataforma, commit, hash da configuração e semente em `artifacts/environment.json`, sem expor credenciais.

In [ ]:
from src.config import log_environment, save_environment_log

# Salva o retrato do ambiente e exibe o conteúdo registrado.
log_path = save_environment_log()
print(f"Registro do ambiente salvo em: {log_path}")
log_environment()